# 04C COF 高通量筛选：模型之后做什么？

> 🔵 **Level B · 建议掌握** | 科研实践 | 完成标准：理解 `ML prediction → candidate list → validation`，而不是把预测排名当成已证明结论。

`reference calculations → surrogate model → candidate library → ranking → applicability domain → validation`


## 1. 研究级 COF screening 的基本结构

```text
COF library
   ↓
structure / pore / chemistry descriptors
   ↓
expensive reference calculations on a subset
   ↓
features + target results
   ↓
train / validation / test
   ↓
ML surrogate
   ↓
predict a much larger candidate library
   ↓
rank candidates → interpret → return to CIF → validate
```

重点不是“Random Forest 比哪个模型更强”，而是 **用少量昂贵计算训练 surrogate，再筛选大量候选结构**。


## 2. 与公开 COF CO₂ screening 仓库对应

可借鉴的仓库：`jsdvos/SupportingInformation_CO2captureHTS_2024`。其中 `Step2_MachineLearning` 使用 `features.csv`、`results.csv`、`structs_train.txt`、`structs_test.txt`，并包含 feature reduction、模型训练和 SHAP 分析。

这给教程一个重要启示：**真实科研项目很少只有一个 notebook。数据生成、模型、筛选和分析应彼此分离，但通过结构 ID 保持可追溯。**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
rng = np.random.default_rng(42)


## 3. 用缩小版 screening table 理解机制

为了让这一章独立运行，我们构造一个**教学 candidate table**。这些 target 是教学公式生成的，不能用于科研结论；真实项目应替换为统一条件下的 GCMC/实验结果。

04B 与 04C 的区别：04B 是 **真实 CIF → descriptor table**；04C 是 **reference subset → surrogate model → large-library screening**。两章合起来才是一条完整路线。


In [ ]:
n = 500
candidates = pd.DataFrame({
    'COF_ID': [f'candidate_{i:04d}' for i in range(n)],
    'PLD_A': rng.uniform(3.0, 18.0, n),
    'LCD_A': rng.uniform(5.0, 35.0, n),
    'ASA_m2_g': rng.uniform(200, 4500, n),
    'void_fraction': rng.uniform(0.25, 0.90, n),
    'density_g_cm3': rng.uniform(0.25, 1.40, n),
    'N_fraction': rng.uniform(0.0, 0.20, n),
    'O_fraction': rng.uniform(0.0, 0.20, n),
})
noise = rng.normal(0, 0.35, n)
candidates['CO2_uptake_demo'] = (
    0.0010*candidates['ASA_m2_g'] + 2.2*candidates['N_fraction']
    + 1.2*candidates['O_fraction'] + 0.8*candidates['void_fraction']
    - 0.045*np.abs(candidates['PLD_A']-7.0) + noise
)
candidates.head()


## 4. 昂贵 reference calculation 只做一部分

假设 500 个候选 COF 中，只负担得起 120 个 reference calculations。机器学习的价值在于从这 120 个学习结构–性质关系，再预测其余材料。


In [ ]:
feature_cols = ['PLD_A','LCD_A','ASA_m2_g','void_fraction','density_g_cm3','N_fraction','O_fraction']
reference = candidates.sample(120, random_state=42).copy()
unseen = candidates.drop(reference.index).copy()
X_train, X_test, y_train, y_test = train_test_split(reference[feature_cols], reference['CO2_uptake_demo'], test_size=0.25, random_state=42)
model = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
test_pred = model.predict(X_test)
print('MAE =', mean_absolute_error(y_test, test_pred))
print('R2  =', r2_score(y_test, test_pred))


## 5. 用 surrogate 筛选未计算的 COF

只有在 test/validation 表现合理之后，才把模型应用到没有 reference target 的 candidate library。


In [ ]:
unseen['predicted_CO2_uptake_demo'] = model.predict(unseen[feature_cols])
top20 = unseen.sort_values('predicted_CO2_uptake_demo', ascending=False).head(20)
display(top20[['COF_ID','predicted_CO2_uptake_demo'] + feature_cols])


## 6. 排名不是终点：检查 applicability domain

预测最高的材料可能位于训练数据范围之外。下面只做最简单的 min–max 检查；科研中可以进一步使用距离、不确定度、ensemble disagreement 等方法。


In [ ]:
train_min = X_train.min()
train_max = X_train.max()
outside = ((top20[feature_cols] < train_min) | (top20[feature_cols] > train_max)).any(axis=1)
top20 = top20.assign(outside_training_range=outside.values)
display(top20[['COF_ID','predicted_CO2_uptake_demo','outside_training_range']])


## 7. 解释：什么特征推动了筛选结果？

公开 COF screening 工作使用 SHAP。初学阶段先从 Random Forest feature importance 看起，再在后续项目升级到 SHAP。


In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
plt.figure(figsize=(6,4))
importance.plot(kind='barh')
plt.xlabel('Random-forest feature importance')
plt.show()


## 8. 真正科研项目还缺哪些步骤？

1. 从 CIF 或专门软件生成可复现 descriptors；
2. 明确 target 条件，例如 CO₂ uptake 的温度、压力、force field/GCMC protocol；
3. 保存 `COF_ID ↔ CIF ↔ feature row ↔ target row`；
4. 去除重复/近重复结构；
5. 设计 random split 以外的 family/topology-aware split；
6. 做超参数选择时避免 test leakage；
7. 用 SHAP/partial dependence 等分析，而不是只报告 R²；
8. 对 top candidates 回到 CIF 检查结构合理性；
9. 对最终候选重新进行高精度模拟或实验验证；
10. 保存模型版本、descriptor 版本与筛选条件。

机器学习筛选的输出应当是**待验证的候选材料**，而不是“模型已经发现了最佳 COF”。


## 9. 推荐继续阅读的仓库路线

- **CURATED-COFs**：真实实验 COF CIF 与结构清理记录。
- **CoRE-COF Database**：更大规模 COF 数据库与版本化结构筛选。
- **SupportingInformation_CO2captureHTS_2024**：CO₂ capture 的高通量 + ML + SHAP 路线。
- **mofdscribe**：虽然主要面向 MOF/多孔材料，但 featurization、benchmark 和 splitting 思想值得迁移到 COF。

不要直接复制仓库脚本。学习重点是识别可复用的科研结构：**data generation → representation → validation → screening → interpretation → verification**。


## Exercises

1. 把 reference 数量改成 40、80、200，比较 test MAE 和 top-20 稳定性。
2. 删除 pore descriptors，只保留 composition；再反过来只保留 pore descriptors。
3. 为 `top20` 设计一个第二阶段昂贵计算队列。
4. 解释为什么不能反复用同一个 test set 调模型又报告它作为最终性能。
5. 写出自己的真实项目目录：`cifs/`、`features/`、`targets/`、`splits/`、`models/`、`predictions/`。

### 04C 完成标准

你不仅会 `model.fit(X, y)`，还能够解释：**结构从哪里来 → descriptor 怎么生成 → target 怎么定义 → 数据怎么对齐 → split 怎么设计 → 模型如何验证 → 如何筛选未知 COF → 为什么还必须回到结构验证。**


## 公开预测表：另一类数据来源
下面读取作者发布的 ML 预测，不是独立模拟标签，也不用于验证上面的人工教学模型。


In [ ]:
import pandas as pd
url='https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/68724HypoCOFs-ML_Predicted%20Data-1bar.csv'
pred=pd.read_csv(url)
print(pred.shape); display(pred.head())
for c in pred.columns:
    if 'CO2' in c.upper() or '1BAR' in c.upper() or '1 BAR' in c.upper(): print(c)


## Screening 不等于排序
还要检查 candidate 是否超出训练域、duplicates、top candidates 是否集中在单一 chemistry family、prediction uncertainty、synthesis/stability constraints，以及是否需要重新做 GCMC/DFT/experiment。

ReDD-COFFEE (`jsdvos/SupportingInformation_CO2captureHTS_2024`) 提供 feature/results tables、固定 train/test、feature reduction、prediction 与 SHAP。


## 3. Dataset C — ReDD-COFFEE CO₂ capture high-throughput screening

`SupportingInformation_CO2captureHTS_2024` 是更接近真实科研工程的案例：

- `features.csv`：大规模 COF descriptors；
- `results.csv`：GCMC / screening targets；
- `structs_train.txt` / `structs_test.txt`：固定数据划分；
- ML scripts：feature reduction、模型训练、prediction 和 SHAP。

完整 feature archive 较大，因此不建议每个 Colab runtime 自动下载全部数据。课程采用两层方式：前两节用小型真实表直接运行；这一节阅读并复现其 **data contract**，需要做大规模项目时再下载完整 ReDD-COFFEE 数据。

官方公开仓库：https://github.com/jsdvos/SupportingInformation_CO2captureHTS_2024


In [ ]:
# results.csv 约数 MB，可直接检查；完整 features archive 更大。
results_url = 'https://raw.githubusercontent.com/jsdvos/SupportingInformation_CO2captureHTS_2024/master/Step2_MachineLearning/data/input/results.csv'
redd_results = pd.read_csv(results_url, sep=';')
print(redd_results.shape)
display(redd_results.head())
print(list(redd_results.columns))


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)


## 用明确列名整理公开预测候选
这些分数来自作者的 COFSpace 模型。导出队列保留 ID、名称和预测来源。范围标记仅与 CoRE 参考表逐列比较，不是校准不确定度，也不等于核验了作者完整训练域。


In [ ]:
prediction_column = 'CO2-1 bar (mol/kg) - ML'
assert {'ID','Name',prediction_column}.issubset(pred.columns)
assert pred['ID'].notna().all() and pred['ID'].is_unique
top_published = pred.dropna(subset=[prediction_column]).nlargest(20, prediction_column).copy()
reference_url = 'https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv'
core_reference = pd.read_csv(reference_url)
domain_cols = ['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O']
top_published['outside_core_reference_range'] = ((top_published[domain_cols] < core_reference[domain_cols].min()) | (top_published[domain_cols] > core_reference[domain_cols].max())).any(axis=1)
top_published['needs_reference_validation'] = True
top_published['prediction_source'] = url
display(top_published[['ID','Name',prediction_column,'outside_core_reference_range']])
top_published.to_csv('published_candidate_queue.csv', index=False)


## 检查作者固定划分
先只下载较小的 train/test 清单，检查互斥和结果覆盖，再考虑大型分卷 feature archive。这里审计数据约定；复现模型仍需作者的特征预处理和训练配置。


In [ ]:
from urllib.request import urlopen
input_base = 'https://raw.githubusercontent.com/jsdvos/SupportingInformation_CO2captureHTS_2024/master/Step2_MachineLearning/data/input/'
def load_ids(filename):
    with urlopen(input_base + filename, timeout=30) as response:
        values = [line.strip() for line in response.read().decode('utf-8').splitlines() if line.strip()]
    assert len(values) == len(set(values)), 'Duplicate split IDs'
    return set(values)
train_ids = load_ids('structs_train.txt')
test_ids = load_ids('structs_test.txt')
assert train_ids.isdisjoint(test_ids), 'Train/test overlap'
result_ids = set(redd_results['struct'].astype(str))
print('train / test IDs:', len(train_ids), len(test_ids))
print('IDs absent from results:', len(train_ids-result_ids), len(test_ids-result_ids))
print('results outside the supplied split:', len(result_ids-train_ids-test_ids))
